## InstanceMethod, ClassMethod, StaticMethod 이론


### `InstanceMethod`
- Python에서 정의하는 일반적인 메서드
- 첫번째 파라미터: `self`
- 인스턴스별로 독립적인 데이터 처리 가능


### `ClassMethod`
- `@classmethod` 데코레이터 사용
- 클래스 자체를 첫 번째 매개변수로 사용 (`cls`)
- 클래스 변수를 수정하거나 클래스 레벨에서 동작화는 메서드를 정의할 때 사용


### `StaticMethod`
- `@staticmethod` 데코레이터 사용
- 인자로 `self`와 `cls`를 **받지 않음**
- 독립적인 유틸리티 함수처럼 사용 가능 (따로 `def`로 사용 가능함)
  - 하지만 staticmethod를 사용하는 이유는 아래와 같다:
    - 클래스와 **관련된 기능**이지만, **인스턴스나 클래스 변수를 사용하지 않는 경우**
    - **코드 가독성 향상**
    - 네임스페이스 관리
          - 해당 기능이 class와 관련된 기능임을 알 수 있게 된다

In [28]:
class Example:
    class_variables : int = 0 # 클래스 변수
    
    def __init__(self, value):
        self.instance_variables = value # 인스턴스 변수

    def instance_method_ex(self):
        """인스턴스 메서드"""
        return f"instance variables: {self.instance_variables}"
        
    @classmethod
    def class_method_ex(cls):
        """클래스 메서드"""
        cls.class_variables += 1
        return f"class variables: {cls.class_variables}"

    @staticmethod
    def static_method_ex(test):
        """스태틱 메서드"""
        return f"static method: {test}"

In [29]:
# 객체 생성
obj1 = Example(10)
obj2 = Example(20)

# 인스턴스 메서드 호출 (인스턴스를 통해 호출)
print(obj1.instance_method_ex())  # Instance Method: value=10
print(obj2.instance_method_ex())  # Instance Method: value=20

# 클래스 메서드 호출 (클래스를 통해 호출)
print(obj1.class_method_ex())  # Class Method: class_variable=1
print(Example.class_method_ex())  # Class Method: class_variable=2 # class 변수 명으로 부를 수 있음

# 정적 메서드 호출 (독립적 호출)
print(Example.static_method_ex(3))  # Static Method: 3 + 5 = 8
print(obj2.static_method_ex(100))  # instance 객체로 부를 수 있음

instance variables: 10
instance variables: 20
class variables: 1
class variables: 2
static method: 3
static method: 100


## Memory of ClassMethod
### ✅ 메모리 관점에서 staticmethod는 어떻게 동작할까?

- `staticmethod`는 클래스 레벨에서만 존재하며, 인스턴스가 생성될 때마다 **새롭게 할당되지 않는다.**
- 즉, **한 번만** 메모리에 로드되며, 인스턴스마다 복사되지 않는다.
- 아래의 예시를 통해 확인을 해볼 수 있을 것 같다.


- 실험 시 아래와 같이 한 줄로 실험을 하면 안되는 현상을 발견함
```python
instance_obj1_id, instance_obj2_id  = id(obj1.instatnce_ex), id(obj2.instatnce_ex)
print(instance_obj1_id == instance_obj2_id) # True 반환
```

In [5]:
class Memory_Ex:
    def instatnce_ex(self):
        return "InstanceMethod Memory Test"
    
    @classmethod
    def class_memory_ex(cls):
        return "ClassMethod Memory Test"
        
    @staticmethod
    def static_memory_ex():
        return "StaticMethod Memory Test"
obj1 = Memory_Ex()
obj2 = Memory_Ex()

#### Instance & Class Method

In [11]:
instance_obj1_id  = id(obj1.instatnce_ex)
instance_obj2_id =  id(obj2.instatnce_ex)
class_obj1_id = id(obj1.class_memory_ex)
class_obj2_id = id(obj2.class_memory_ex)

In [12]:
print(instance_obj1_id == instance_obj2_id)
print(class_obj1_id == class_obj2_id)

False
False


#### Static Method

In [13]:
static_obj1_id = id(obj1.static_memory_ex)
static_obj2_id = id(obj2.static_memory_ex)
print(static_obj1_id)
print(static_obj2_id)

# 두개의 메모리 주소 값이 같은지 확인
print(static_obj1_id == static_obj2_id) 

139725043390464
139725043390464
True


### Bound & Unbound Method

- 이제 여기서 `bound, unbound method`가 등장한다.

#### Bound Method?
- 특정 인스턴스에 바인딩된(연결된) 메서드를 의미
- 어떤 멤버 함수가 어떤 클래스에 속해 있는 method라는 것을 의미할 때, 이것을 `bound method`라고 함

#### Unbound Method?
- 특정 인스턴스에 바인딩되지 않은 메서드를 의미
- self나 cls 없이 호출되는 메서드를 의미한다
- staticmethod가 아닐 때도 아래와 같이 사용하면 unbound method가 된다.
  ```python
  class Ex:
      def test():
          return 'test'
  ```
  - 해당 test method는 호출하기 위하여 `static method`로 변환해주어야 한다.

In [14]:
class Demo:
    def instance_method(self):
        return "I'm an instance method"
    
    @staticmethod
    def static_method():
        return "I'm a static method"

    @classmethod
    def class_method(cls):
        return "I'm a class method"

# 인스턴스 생성
obj = Demo()

# 인스턴스 메서드 (Bound Method)
print(obj.instance_method)  # <bound method Demo.instance_method of <__main__.Demo object at 0x...>>
print(obj.instance_method())  # I'm an instance method

# 클래스에서 직접 호출하면 일반 함수처럼 동작 (Unbound Method)
print(Demo.instance_method)  # <function Demo.instance_method at 0x...> (Unbound)

# 정적 메서드 (일반 함수처럼 호출됨)
print(Demo.static_method)  # <function Demo.static_method at 0x...> (Unbound)
print(obj.static_method)  # <function Demo.static_method at 0x...> (Unbound)

# 클래스 메서드 (Bound to class, not instance)
print(Demo.class_method)  # <bound method Demo.class_method of <class '__main__.Demo'>>
print(obj.class_method)  # <bound method Demo.class_method of <class '__main__.Demo'>>

<bound method Demo.instance_method of <__main__.Demo object at 0x7f14461ef530>>
I'm an instance method
<function Demo.instance_method at 0x7f144593d120>
<function Demo.static_method at 0x7f144593dbc0>
<function Demo.static_method at 0x7f144593dbc0>
<bound method Demo.class_method of <class '__main__.Demo'>>
<bound method Demo.class_method of <class '__main__.Demo'>>
